# 🩺 VPIC — Training, Evaluation & Testing
## YOLOv8n Syringe Detection

**Run `VPIC_DataPipeline.ipynb` first** to generate `vpic_dataset/` before using this notebook.

| Stage | Description |
|---|---|
| 1 | Setup & GPU check |
| 2 | Train YOLOv8n |
| 3 | Training curves |
| 4 | Validation evaluation (mAP, PR curve, confusion matrix) |
| 5 | Test set inference & visualisation |
| 6 | Error analysis |
| 7 | Export to ONNX (Quest 3 / Vision Pro) |

---
## 1. Setup & GPU Check

In [ ]:
%matplotlib inline

import sys, random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import torch
from ultralytics import YOLO

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('.').resolve()               # VPIC/
DATASET_DIR = BASE_DIR / 'vpic_dataset'
YAML_PATH   = DATASET_DIR / 'dataset.yaml'
RUNS_DIR    = BASE_DIR / 'runs'                 # YOLOv8 saves results here

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_SIZE  = 'yolov8n.pt'    # nano — fastest for real-time AR/VR
EPOCHS      = 100
IMGSZ       = 640
BATCH       = 16
RUN_NAME    = 'vpic_syringe_v1'
PATIENCE    = 20              # early stopping
CLASS_NAMES = ['syringe']
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'

print(f'Python     : {sys.version.split()[0]}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
print(f'Device     : {DEVICE}')
print(f'Dataset    : {YAML_PATH}  exists={YAML_PATH.exists()}')
print(f'Model      : {MODEL_SIZE}')

# Dataset split sizes
for split in ('train', 'val', 'test'):
    n = len(list((DATASET_DIR / 'images' / split).iterdir()))
    print(f'  {split:6s}: {n} images')

---
## 2. Train YOLOv8n

Fine-tunes the YOLOv8 nano pretrained weights on your syringe dataset.  
Training logs and weights are saved to `runs/detect/<RUN_NAME>/`.

> **Tip:** On CPU this will take a while. If you have a GPU, training 100 epochs typically takes ~5–15 minutes.

In [ ]:
model = YOLO(MODEL_SIZE)

results = model.train(
    data     = str(YAML_PATH),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    name     = RUN_NAME,
    patience = PATIENCE,
    device   = DEVICE,
    project  = str(RUNS_DIR),
    exist_ok = True,          # overwrite if re-running
    # Augmentation (Albumentations handled in pipeline; YOLOv8 also does its own)
    hsv_h    = 0.015,         # hue jitter
    hsv_s    = 0.5,           # saturation jitter
    hsv_v    = 0.4,           # brightness jitter
    flipud   = 0.0,           # no vertical flip — needle direction matters
    fliplr   = 0.5,           # horizontal flip OK
    mosaic   = 0.5,           # mosaic augmentation
    translate= 0.1,
    scale    = 0.3,
)

# Path to this run's output folder
RUN_DIR = RUNS_DIR / 'detect' / RUN_NAME
print(f'\nTraining complete.')
print(f'Results saved to: {RUN_DIR}')

---
## 3. Training Curves

Plots loss and mAP over all epochs from YOLOv8's `results.csv`.

In [ ]:
RUN_DIR = RUNS_DIR / 'detect' / RUN_NAME   # re-define in case you skipped cell 2
csv_path = RUN_DIR / 'results.csv'

if not csv_path.exists():
    print(f'results.csv not found at {csv_path}')
    print('Run the training cell first.')
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # strip whitespace from column names

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f'Training Curves — {RUN_NAME}', fontsize=13)

    def plot(ax, col, title, color):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=1.5)
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.set_title(f'{title} (not found)', fontsize=9)
            ax.axis('off')

    plot(axes[0,0], 'train/box_loss',  'Train Box Loss',       'steelblue')
    plot(axes[0,1], 'train/cls_loss',  'Train Class Loss',     'salmon')
    plot(axes[0,2], 'train/dfl_loss',  'Train DFL Loss',       'mediumpurple')
    plot(axes[1,0], 'val/box_loss',    'Val Box Loss',         'dodgerblue')
    plot(axes[1,1], 'metrics/mAP50',   'mAP@0.5',              'mediumseagreen')
    plot(axes[1,2], 'metrics/mAP50-95','mAP@0.5:0.95',         'darkorange')

    plt.tight_layout()
    plt.savefig(RUN_DIR / 'training_curves.png', dpi=150)
    plt.show()

    # Best epoch summary
    if 'metrics/mAP50' in df.columns:
        best = df.loc[df['metrics/mAP50'].idxmax()]
        print(f'Best epoch    : {int(best["epoch"])}')
        print(f'mAP@0.5       : {best["metrics/mAP50"]:.4f}')
        print(f'mAP@0.5:0.95  : {best["metrics/mAP50-95"]:.4f}')

---
## 4. Validation Evaluation

Runs the best checkpoint against the **val** split and reports mAP, precision, recall, and plots the PR curve and confusion matrix.

In [ ]:
best_weights = RUN_DIR / 'weights' / 'best.pt'

if not best_weights.exists():
    print(f'best.pt not found at {best_weights}')
    print('Run training first.')
else:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    eval_model  = YOLO(str(best_weights))
    try:
        val_results = eval_model.val(
            data    = str(YAML_PATH),
            split   = 'val',
            imgsz   = IMGSZ,
            device  = DEVICE,
            project = str(RUNS_DIR),
            name    = RUN_NAME + '_val',
            exist_ok= True,
        )
    except RuntimeError as e:
        if 'CUDA' in str(e):
            print(f'CUDA error: {e}')
            print('Retrying on CPU...')
            val_results = eval_model.val(
                data    = str(YAML_PATH),
                split   = 'val',
                imgsz   = IMGSZ,
                device  = 'cpu',
                project = str(RUNS_DIR),
                name    = RUN_NAME + '_val',
                exist_ok= True,
            )
        else:
            raise

    print('\n' + '═' * 45)
    print('  Validation Results')
    print('─' * 45)
    print(f'  mAP@0.5       : {val_results.box.map50:.4f}')
    print(f'  mAP@0.5:0.95  : {val_results.box.map:.4f}')
    print(f'  Precision     : {val_results.box.mp:.4f}')
    print(f'  Recall        : {val_results.box.mr:.4f}')
    print('═' * 45)


In [ ]:
# ── PR Curve (saved by ultralytics, display here) ─────────────────────────────
val_run_dir = RUNS_DIR / (RUN_NAME + '_val')
pr_img_path = val_run_dir / 'PR_curve.png'

if pr_img_path.exists():
    img = plt.imread(str(pr_img_path))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Precision–Recall Curve', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f'PR curve image not found at {pr_img_path}')
    print('It is generated automatically after running the val cell above.')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
cm_path = val_run_dir / 'confusion_matrix_normalized.png'
if not cm_path.exists():
    cm_path = val_run_dir / 'confusion_matrix.png'

if cm_path.exists():
    img = plt.imread(str(cm_path))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Confusion Matrix', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f'Confusion matrix not found at {cm_path}')

---
## 5. Test Set Inference & Visualisation

Runs the best model on the held-out **test** split and overlays predictions vs ground truth side by side.

In [ ]:
TEST_IMAGES_DIR = DATASET_DIR / 'images' / 'test'
TEST_LABELS_DIR = DATASET_DIR / 'labels' / 'test'
CONF_THRESHOLD  = 0.25    # detection confidence threshold
N_DISPLAY       = 8       # number of test frames to show

if not best_weights.exists():
    print('Train the model first.')
else:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    infer_model = YOLO(str(best_weights))
    test_images = sorted(TEST_IMAGES_DIR.iterdir())
    sample_imgs = random.sample(test_images, min(N_DISPLAY, len(test_images)))

    cols = 4
    rows = (len(sample_imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = np.array(axes).flatten()

    PRED_COLOR = (0.20, 0.85, 0.40)   # green — prediction
    GT_COLOR   = (1.00, 0.40, 0.40)   # red   — ground truth

    for ax, img_path in zip(axes, sample_imgs):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # ── Run inference ───────────────────────────────────────────────────
        preds = infer_model(img_bgr, conf=CONF_THRESHOLD, verbose=False)[0]

        ax.imshow(img_rgb)

        # ── Draw ground truth (red) ─────────────────────────────────────────
        label_path = TEST_LABELS_DIR / (img_path.stem + '.txt')
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) != 5: continue
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                ax.add_patch(patches.Rectangle(
                    (x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor=GT_COLOR, facecolor='none',
                    linestyle='--'
                ))
                ax.text(x1, max(y1-14, 14), 'GT', color=GT_COLOR, fontsize=8,
                        bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))

        # ── Draw predictions (green) ────────────────────────────────────────
        for box in preds.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=PRED_COLOR, facecolor='none'
            ))
            ax.text(x1, max(y1-2, 2), f'syringe {conf:.2f}',
                    color=PRED_COLOR, fontsize=8, fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))

        ax.set_title(img_path.stem, fontsize=7)
        ax.axis('off')

    for ax in axes[len(sample_imgs):]:
        ax.axis('off')

    fig.legend(
        handles=[
            patches.Patch(edgecolor=GT_COLOR,   facecolor='none', linestyle='--', label='Ground Truth'),
            patches.Patch(edgecolor=PRED_COLOR, facecolor='none', label='Prediction'),
        ],
        loc='lower center', ncol=2, fontsize=10, framealpha=0.8
    )
    plt.suptitle(f'Test Set Predictions — conf≥{CONF_THRESHOLD}', fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(RUN_DIR / 'test_predictions.png', dpi=150)
    plt.show()
    print(f'Saved to: {RUN_DIR / "test_predictions.png"}')

---
## 6. Test Set Metrics & Error Analysis

Runs the full test split through the model and computes per-image stats.  
Identifies **missed detections** (false negatives) and **false positives**.

In [ ]:
# ── Official test-split evaluation ────────────────────────────────────────────
if best_weights.exists():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    test_model = YOLO(str(best_weights))
    try:
        test_results = test_model.val(
            data     = str(YAML_PATH),
            split    = 'test',
            imgsz    = IMGSZ,
            device   = DEVICE,
            project  = str(RUNS_DIR),
            name     = RUN_NAME + '_test',
            exist_ok = True,
        )
    except RuntimeError as e:
        if 'CUDA' in str(e):
            print(f'CUDA error: {e}')
            print('Retrying on CPU...')
            test_results = test_model.val(
                data     = str(YAML_PATH),
                split    = 'test',
                imgsz    = IMGSZ,
                device   = 'cpu',
                project  = str(RUNS_DIR),
                name     = RUN_NAME + '_test',
                exist_ok = True,
            )
        else:
            raise

    print('\n' + '═' * 45)
    print('  Test Set Results')
    print('─' * 45)
    print(f'  mAP@0.5       : {test_results.box.map50:.4f}')
    print(f'  mAP@0.5:0.95  : {test_results.box.map:.4f}')
    print(f'  Precision     : {test_results.box.mp:.4f}')
    print(f'  Recall        : {test_results.box.mr:.4f}')
    print('═' * 45)


In [ ]:
# ── Per-image error analysis ───────────────────────────────────────────────────
IOU_THRESH = 0.5

def iou(boxA, boxB):
    """Compute IoU between two [x1,y1,x2,y2] boxes."""
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

if best_weights.exists():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    infer_model = YOLO(str(best_weights))
    test_images = sorted(TEST_IMAGES_DIR.iterdir())

    missed, false_pos, correct = [], [], []

    for img_path in test_images:
        img_bgr = cv2.imread(str(img_path))
        h, w    = img_bgr.shape[:2]
        preds   = infer_model(img_bgr, conf=CONF_THRESHOLD, verbose=False)[0]

        # Ground truth boxes in xyxy
        label_path = TEST_LABELS_DIR / (img_path.stem + '.txt')
        gt_boxes = []
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                p = line.split()
                if len(p) == 5:
                    cx, cy, bw, bh = map(float, p[1:])
                    gt_boxes.append([
                        (cx-bw/2)*w, (cy-bh/2)*h,
                        (cx+bw/2)*w, (cy+bh/2)*h
                    ])

        pred_boxes = [box.xyxy[0].cpu().numpy().tolist() for box in preds.boxes]

        # Match preds to GT
        matched_gt = set()
        for pb in pred_boxes:
            best_iou, best_idx = 0, -1
            for i, gb in enumerate(gt_boxes):
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_idx = v, i
            if best_iou >= IOU_THRESH and best_idx not in matched_gt:
                matched_gt.add(best_idx)
                correct.append(img_path)
            else:
                false_pos.append(img_path)

        for i in range(len(gt_boxes)):
            if i not in matched_gt:
                missed.append(img_path)

    print(f'Test images   : {len(test_images)}')
    print(f'Correct detections (IoU≥{IOU_THRESH}): {len(correct)}')
    print(f'False positives : {len(false_pos)}')
    print(f'Missed (FN)     : {len(missed)}')

In [ ]:
# ── Display missed detections ──────────────────────────────────────────────────
def show_frames(img_paths, title, n=6):
    if not img_paths:
        print(f'No {title} — great!')
        return
    samples = random.sample(list(set(img_paths)), min(n, len(set(img_paths))))
    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4))
    axes = np.array(axes).flatten()
    for ax, p in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(p.stem, fontsize=7)
        ax.axis('off')
    for ax in axes[len(samples):]:
        ax.axis('off')
    plt.suptitle(title, fontsize=11, color='crimson')
    plt.tight_layout()
    plt.show()

show_frames(missed,    'Missed Detections (False Negatives)')
show_frames(false_pos, 'False Positives')

---
## 7. Export to ONNX

Exports the best weights to ONNX format for deployment on:
- **Meta Quest 3** — via Unity Barracuda or ONNX Runtime
- **Apple Vision Pro** — via Core ML conversion (run `coremltools` on the ONNX)
- **PC inference** — ONNX Runtime CPU/GPU

In [ ]:
if best_weights.exists():
    export_model = YOLO(str(best_weights))

    # ── ONNX export ───────────────────────────────────────────────────────────
    onnx_path = export_model.export(
        format  = 'onnx',
        imgsz   = IMGSZ,
        dynamic = False,     # static input shape — needed for Quest 3 / Barracuda
        simplify= True,      # simplify ONNX graph
        opset   = 12,        # opset 12 for widest compatibility
    )
    print(f'\nONNX model saved to: {onnx_path}')

    # ── Verify ONNX loads correctly ───────────────────────────────────────────
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(onnx_path))
        inp  = sess.get_inputs()[0]
        out  = sess.get_outputs()[0]
        print(f'ONNX input  : {inp.name}  shape={inp.shape}')
        print(f'ONNX output : {out.name}  shape={out.shape}')
        print('ONNX verification ✅')
    except ImportError:
        print('onnxruntime not installed — run: pip install onnxruntime')
    except Exception as e:
        print(f'ONNX verification failed: {e}')
else:
    print('Train the model first.')

In [ ]:
# ── Apple Vision Pro — Core ML conversion (optional) ─────────────────────────
# Uncomment and run after ONNX export to get a .mlpackage for Vision Pro.
#
# pip install coremltools
#
# import coremltools as ct
# mlmodel = ct.convert(
#     str(onnx_path),
#     inputs=[ct.TensorType(shape=(1, 3, 640, 640))],
#     minimum_deployment_target=ct.target.iOS16,
# )
# mlmodel.save(str(RUN_DIR / 'vpic_syringe.mlpackage'))
# print('Core ML model saved.')

---
## ✅ Done

| Output file | Location |
|---|---|
| Best weights | `runs/detect/vpic_syringe_v1/weights/best.pt` |
| Training curves | `runs/detect/vpic_syringe_v1/training_curves.png` |
| Test predictions | `runs/detect/vpic_syringe_v1/test_predictions.png` |
| ONNX model | `runs/detect/vpic_syringe_v1/weights/best.onnx` |

### Next steps
- Load `best.pt` in a new `VPIC_Inference.ipynb` for real-time stereo camera tracking
- Drop `best.onnx` into your Unity project (Barracuda) for Quest 3 integration
